# Module 1.2 - LLM Fundamentals

**Reference:** [`02-llm-fundamentals.md`](./02-llm-fundamentals.md)

## What you'll do in this notebook

- Use `tiktoken` to inspect how text is broken into tokens.
- Estimate the token cost of a conversation as it grows.
- See how `temperature`, `top_p`, and `max_tokens` change the model's output.
- Detect when a response was truncated because it hit `max_tokens`.

## Setup

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import tiktoken

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set."

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
encoding = tiktoken.encoding_for_model("gpt-4o-mini")
print(f"OK: client ready, model = {MODEL}")

## Exercise 1 - Tokenisation

Tokens are not words. Use `encoding.encode(text)` to get the list of token IDs and `encoding.decode([id])` to recover the string for any single token. Run the cell below and then complete the TODOs.

In [ ]:
samples = [
    "Hello, world!",
    "Chatbot development",
    "AI",
    "antidisestablishmentarianism",
    "The quick brown fox jumps over the lazy dog.",
]

for text in samples:
    token_ids = encoding.encode(text)
    pieces = [encoding.decode([tid]) for tid in token_ids]
    print(f"{text!r:50} -> {len(token_ids)} tokens: {pieces}")

In [ ]:
# TODO: write a function token_count(text) -> int that returns the number of tokens in text.
def token_count(text: str) -> int:
    raise NotImplementedError

# TODO: using token_count, find an English word (real or nonsense) that is exactly 1 token long
# and another that is at least 4 tokens long. Print both with their token counts.


## Exercise 2 - Context-window budgeting

Consider a conversation where:

- The system prompt is ~100 tokens.
- Each user message is ~50 tokens.
- Each assistant reply is ~150 tokens.

The full conversation history is sent on every API call, so tokens accumulate. Fill in the TODOs to compute the running token total after each turn, and identify the turn at which a 16,000-token context window would be exhausted.

In [ ]:
SYSTEM_PROMPT_TOKENS = 100
USER_TOKENS_PER_TURN = 50
ASSISTANT_TOKENS_PER_TURN = 150
CONTEXT_WINDOW = 16_000

# TODO: for turns 1..60, compute the total number of tokens that would be sent
# on the API call for that turn, print a table row, and stop once you cross CONTEXT_WINDOW.
# Hint: the prompt on turn N includes the system prompt + every prior user/assistant pair +
# the new user message.


## Exercise 3 - Temperature and top_p

Run the cell below three times at `temperature=0.0` and three times at `temperature=1.5`. What do you notice about variability and coherence?

In [ ]:
PROMPT = "Invent a single-sentence slogan for a cafe called DevBrew that serves programmers."

def sample_once(temperature: float) -> str:
    # TODO: call client.chat.completions.create with:
    #   - model=MODEL
    #   - messages containing a user turn with PROMPT
    #   - the given temperature
    # Return the assistant's reply.
    raise NotImplementedError

print("-- temperature = 0.0 --")
for _ in range(3):
    print("-", sample_once(0.0))

print("\n-- temperature = 1.5 --")
for _ in range(3):
    print("-", sample_once(1.5))

**Reflect:**

- For a customer-support bot, what temperature would you pick and why?
- For a creative-writing assistant, what would you pick?

## Exercise 4 - max_tokens and finish_reason

Set `max_tokens` very low and ask for a long response. The reply will be cut off mid-sentence, and `finish_reason` on the response will be `"length"` rather than `"stop"`. Production code should check this.

In [ ]:
# TODO: call the model with max_tokens=20, asking it to "explain how transformers work in detail".
# Print:
#   - the truncated reply
#   - the finish_reason from the response
#   - the prompt_tokens, completion_tokens, and total_tokens from response.usage


## What's next

In the next section (`03-api-integration-basics.ipynb`) we use these fundamentals in a first full chatbot loop: sending system + user messages, inspecting the response object, and wrapping the call in retry logic that survives rate limits.